In [1]:
import os
import pandas as pd
from hashlib import md5
from PIL import Image

In [4]:
baseline_data_path = "/Users/lukasb/Documents/data/surfaceClassification/baseline"

artifacts_path = "./artifacts/baseline"

### Copy near duplicates to separate folder for manual labeling

In [5]:
from shutil import copy2

df_path = os.path.join(artifacts_path, "baseline_images.csv")
df = pd.read_csv(df_path)
print(f"Loaded baseline images dataframe with {len(df)} entries from {df_path}")

similarity_path = os.path.join(artifacts_path, "similarity_pairs.csv")
similarity_df = pd.read_csv(similarity_path)
print(f"Loaded {len(similarity_df)} similarity pairs from {similarity_path}")

high_threshold = 0.99
medium_threshold = 0.92
low_threshold = 0.85

# Filter by thresholds
high_sim = similarity_df[similarity_df["similarity"] >= high_threshold]
medium_sim = similarity_df[(similarity_df["similarity"] >= medium_threshold) & 
                           (similarity_df["similarity"] < high_threshold)]
low_sim = similarity_df[(similarity_df["similarity"] >= low_threshold) & 
                        (similarity_df["similarity"] < medium_threshold)]

print(f"High similarity pairs (>= {high_threshold}): {len(high_sim)}")
print(f"Medium similarity pairs ({medium_threshold} - {high_threshold}): {len(medium_sim)}")
print(f"Low similarity pairs ({low_threshold} - {medium_threshold}): {len(low_sim)}")

# Sample 150 pairs from each category
n_samples = min(150, len(high_sim), len(medium_sim), len(low_sim))
high_sample = high_sim.sample(n=min(n_samples, len(high_sim)), random_state=42)
medium_sample = medium_sim.sample(n=min(n_samples, len(medium_sim)), random_state=42)
low_sample = low_sim.sample(n=min(n_samples, len(low_sim)), random_state=42)

# Create output directory
output_dir = os.path.join(artifacts_path, "cluster_samples")
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/high", exist_ok=True)
os.makedirs(f"{output_dir}/medium", exist_ok=True)
os.makedirs(f"{output_dir}/low", exist_ok=True)

# Copy sampled image pairs and create pair manifest

def copy_pair(row, idx, category):
    img1_rel = df[df["image_id"] == row["image_id_1"]]["file_path"].values[0]
    img2_rel = df[df["image_id"] == row["image_id_2"]]["file_path"].values[0]
    img1_path = os.path.join(baseline_data_path, img1_rel.lstrip(os.sep))
    img2_path = os.path.join(baseline_data_path, img2_rel.lstrip(os.sep))
    
    ext1 = os.path.splitext(img1_path)[1]
    ext2 = os.path.splitext(img2_path)[1]
    
    copy2(img1_path, f"{output_dir}/{category}/pair_{idx:03d}_img1{ext1}")
    copy2(img2_path, f"{output_dir}/{category}/pair_{idx:03d}_img2{ext2}")

# Create manifest for sampled pairs
sampled_pairs = []
pair_counter = 0

for idx, row in high_sample.iterrows():
    copy_pair(row, pair_counter, "high")
    sampled_pairs.append({
        "pair_id": f"high_pair_{pair_counter:03d}",
        "image_id_a": row["image_id_1"],
        "image_id_b": row["image_id_2"],
        "similarity": row["similarity"],
        "category": "high"
    })
    pair_counter += 1

pair_counter = 0
for idx, row in medium_sample.iterrows():
    copy_pair(row, pair_counter, "medium")
    sampled_pairs.append({
        "pair_id": f"medium_pair_{pair_counter:03d}",
        "image_id_a": row["image_id_1"],
        "image_id_b": row["image_id_2"],
        "similarity": row["similarity"],
        "category": "medium"
    })
    pair_counter += 1

pair_counter = 0
for idx, row in low_sample.iterrows():
    copy_pair(row, pair_counter, "low")
    sampled_pairs.append({
        "pair_id": f"low_pair_{pair_counter:03d}",
        "image_id_a": row["image_id_1"],
        "image_id_b": row["image_id_2"],
        "similarity": row["similarity"],
        "category": "low"
    })
    pair_counter += 1

# Save sampled pairs manifest
sampled_pairs_df = pd.DataFrame(sampled_pairs)
sampled_pairs_df.to_csv(f"{output_dir}/sampled_pairs.csv", index=False)

print(f"Copied {len(high_sample)} high similarity pairs")
print(f"Copied {len(medium_sample)} medium similarity pairs")
print(f"Copied {len(low_sample)} low similarity pairs")
print(f"Saved sampled pairs manifest to {output_dir}/sampled_pairs.csv")

Loaded baseline images dataframe with 5586 entries from ./artifacts/baseline/baseline_images.csv
Loaded 79615 similarity pairs from ./artifacts/baseline/similarity_pairs.csv
High similarity pairs (>= 0.99): 284
Medium similarity pairs (0.92 - 0.99): 64241
Low similarity pairs (0.85 - 0.92): 14635
Copied 150 high similarity pairs
Copied 150 medium similarity pairs
Copied 150 low similarity pairs
Saved sampled pairs manifest to ./artifacts/baseline/cluster_samples/sampled_pairs.csv


# LABEL THE IMAGES MANUALLY USING THE STREAMLIT WEB APP